In [1]:
import _referAsMain
from pathlib import Path
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint

import torch
from torch.utils.data import DataLoader

from dataset import svg_dataset
from paths_cfg import OUR_DATASET_DIRECTORY, TOKENIZER_SAVE_DIRECTORY
import metrics.chunck as chunck
import tokenizer_pfe.tokenizer_project as tokenizerLib

added '/home/holo/dev/PFE_LLM_art_generation' to import paths


In [2]:
tokenizer_path = TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer")
tokenizer = tokenizerLib.Tokenizer.load(tokenizer_path)

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json


In [3]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=32,
    tokenizer=tokenizer.encode, decoder=tokenizer.decode)

In [4]:
dataset4096 = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=4096,
    tokenizer=tokenizer.encode, decoder=tokenizer.decode)

In [5]:
for i in range(3):
    print(f"Chunk {i}")
    print(dataset.chunks[i].text[:40])
    print()

Chunk 0
<svg xmlns:xlink="http://www.w3.org/1999

Chunk 1
="http://www.w3.org/2000/svg" style="fil

Chunk 2
:1; color-rendering:auto; color-interpol



In [6]:
N = 0
for i in range(10):
    print(" "*N, dataset.chunks[i].text, sep="")
    N += len(dataset.chunks[i].text)//2


<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity
                                                   ="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke
                                                                                                                   :1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke
                                                                                                                                                                                                   :black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke
                                                                                           

In [37]:
import importlib
import metrics.chunck
_ = importlib.reload(metrics.chunck)
import metrics.chunck as chunck

In [8]:
#import importlib
#import metrics.chunck
#_ = importlib.reload(metrics.chunck)
#import metrics.chunck as chunck

chunk_tokens = dataset.chunks[0].tokens
#chunk_tokens = torch.tensor(list(range(256)))
seq_len = len(chunk_tokens)
vocab_size = tokenizer.get_vocab_size()
print(seq_len, vocab_size)
chunk_logits = torch.full((1, seq_len, vocab_size), -torch.inf)
#chunk_logits = torch.randn(1, seq_len, vocab_size)
for i, tk in enumerate(chunk_tokens):
    chunk_logits[0, i, tk] = 10.0


tokens_sampled = chunck.sampling_logits(chunk_logits, top_k=None, temperature=1.0)
print("Tokens of original :", chunk_tokens[: 200].tolist())
print("Tokens after sampling :", tokens_sampled[: 200])
print(f"nb diffs: {sum([a!=b for a,b in zip(chunk_tokens.tolist(), tokens_sampled)])}")

32 585
Tokens of original : [519, 487, 518, 258, 491, 465, 492, 460, 20, 496, 16, 328, 365, 514, 3, 487, 258, 491, 465, 492, 460, 20, 496, 16, 382, 386, 513, 3, 269, 258, 273, 503]
Tokens after sampling : [519, 487, 518, 258, 491, 465, 492, 460, 20, 496, 16, 328, 365, 514, 3, 487, 258, 491, 465, 492, 460, 20, 496, 16, 382, 386, 513, 3, 269, 258, 273, 503]
nb diffs: 0


Ci-dessous les tests avec de multiples fichiers svg

In [ ]:
svgs_text, svgs_tokens = chunck.assemble_decode(
    dataset=dataset,
    tokenizer=tokenizer,
    context_size=dataset.context_size,
    vocab_size=vocab_size,
    temperature=1.0,
    top_k=None,
    batch_size= 10000
)

# 3️⃣ Comparer avec l'original
for svg_idx, text in svgs_text.items():
    original_svg = dataset.samples[svg_idx].txt
    print(f"SVG {svg_idx}: {'meme' if text == original_svg else 'different'}")

0 is decoded
1 is decoded
2 is decoded
3 is decoded
4 is decoded
5 is decoded
6 is decoded
7 is decoded
8 is decoded
9 is decoded
10 is decoded
11 is decoded
12 is decoded
13 is decoded
14 is decoded
15 is decoded
16 is decoded
17 is decoded
18 is decoded
19 is decoded
20 is decoded
21 is decoded
22 is decoded
23 is decoded
24 is decoded
25 is decoded
26 is decoded
27 is decoded
28 is decoded
29 is decoded
30 is decoded
31 is decoded
32 is decoded
33 is decoded
34 is decoded
35 is decoded
36 is decoded
37 is decoded
38 is decoded
39 is decoded
40 is decoded
41 is decoded
42 is decoded
43 is decoded
44 is decoded
45 is decoded
46 is decoded
47 is decoded
48 is decoded
49 is decoded
50 is decoded
51 is decoded
52 is decoded
53 is decoded
54 is decoded
55 is decoded
56 is decoded
57 is decoded
58 is decoded
59 is decoded
60 is decoded
61 is decoded
62 is decoded
63 is decoded
64 is decoded
65 is decoded
66 is decoded
67 is decoded
68 is decoded
69 is decoded
70 is decoded
71 is decoded
72

## my tests

In [ ]:

testDevice = torch.device("cuda:0")

prof = Profiler(["iterDataloader", "split", "target", "logits", "print"])

dataloader = DataLoader(dataset4096, batch_size=32, shuffle=True, num_workers=0)
assembler = chunck.ChunckAssembler(
    tokenizer=tokenizer, context_size=dataset.context_size,
    temperature=0.0, top_k=5)

print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader))
while True:
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"]
        assert isinstance(tokens, torch.Tensor)
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()

    with prof.mesure("target"):
        if True:
            target = tokens
        else: target = torch.tensor(list(range(256)), dtype=torch.int64).view(1, -1)
        target = target.to(testDevice)

    with prof.mesure("logits"):
        batchSize, seq_len = target.shape
        vocab_size = tokenizer.get_vocab_size()
        if True:
            logits = torch.full((batchSize, seq_len, vocab_size), -10.0, device=testDevice)
            mask = (target != svg_dataset.IGNORE_INDEX)
            b, t = mask.nonzero(as_tuple=True)
            logits[b, t, target[b, t]] = 0.0
            logits += 0.05 * torch.randn(batchSize, seq_len, vocab_size, device=testDevice)
        else: logits = torch.randn(batchSize, seq_len, vocab_size, device=testDevice)# * 0.001

    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    # print(f"target of shape: {target.shape}[{target.dtype}]")
    
    assembler.add_logits(logits, svgIndexes=svgIndex, chunckIndexes=chunckIndex)

print()
res = assembler.assemble_chuncks()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(assembler._prof.pretty_totalTimes())

there are 432 batch to process
doing: i=430
{
    iterDataloader: 199.515 ms,
    split: 10.656 ms,
    target: 67.104 ms,
    logits: 657.602 ms,
    print: 59.944 ms
}
{
    sample: 10.186 sec,
    sample1: 1.347 sec,
    sample2.2: 521.498 ms,
    sample3: 7.843 sec,
    decode: 10.861 sec,
    assemble: 682.324 ms
}
{
    sample: 0.739 ms,
    sample1: 97.623 μs,
    sample2.2: 37.809 μs,
    sample3: 0.569 ms
}
